In [5]:
import pandas as pd

df = pd.read_csv("../data_processed/df_pumps_prepared.csv")

In [6]:
df.head()

,tertiary_class,object_key,distance_floor,distance_mtr,duration_sec,segment_count,stay_duration_sec,date_string,hour_of_day,minute_of_hour,stay_duration_hours
0,Object 9,131846,2,64.28,60.0,2,4805.0,13.08.2025,11,4,1.334722
1,Object 9,131846,0,30.30,0.0,1,358955.0,17.08.2025,14,47,99.709722
2,Object 9,131847,0,49.32,0.0,1,338451.0,30.08.2025,11,15,94.014167
3,Object 9,131851,0,59.10,119.0,2,439.0,30.08.2025,15,56,0.121944
4,Object 9,131851,0,108.27,180.0,2,1182.0,31.08.2025,18,23,0.328333


In [7]:
# Combine date, hour, and minute into a single timestamp (event_time)
df["event_time"] = pd.to_datetime(
    df["date_string"],
    dayfirst=True
) + pd.to_timedelta(df["hour_of_day"], unit="h") \
  + pd.to_timedelta(df["minute_of_hour"], unit="m")

In [8]:
# Sort data by device and time (required to preserve temporal order)
df = df.sort_values(["object_key", "event_time"])

In [34]:
# Get the timestamp of the next event for each device
df["next_event_time"] = df.groupby("object_key")["event_time"].shift(-1)

# Calculate time (in hours) until the next event
df["time_to_next"] = (
    (df["next_event_time"] - df["event_time"])
    .dt.total_seconds() / 3600
)

# Define target: 1 if next event occurs within 12 hours, else 0
df["target"] = (df["time_to_next"] <= 6).astype(int)

# Remove last observation per device (no next event available)
df = df.dropna(subset=["time_to_next"])

In [35]:
# Select simple and valid features (available at prediction time)
features = [
    "stay_duration_hours",  # time since last movement
    "hour_of_day"           # time-of-day effect
]

In [36]:
# Split data based on time (avoid random split to preserve temporal structure)
train = df[df["event_time"] < "2025-08-25"]
test  = df[df["event_time"] >= "2025-08-25"]

In [37]:
from sklearn.linear_model import LogisticRegression

# Define inputs (X) and target (y)
X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict probabilities for test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [38]:
from sklearn.metrics import roc_auc_score

# Compute AUC (model performance)
auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)

AUC: 0.606704400510204


In [19]:
import numpy as np

# Estimate share of equipment likely to remain inactive (idle)
idle_share = np.mean(y_pred_proba < 0.6)

print("Idle share:", idle_share)

Idle share: 0.006224066390041493


## 3 simple improvements

In [21]:
features = [
    "stay_duration_hours",
    "hour_of_day",
    "minute_of_hour"
]

In [22]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]

In [23]:
import numpy as np

df["log_stay_duration"] = np.log1p(df["stay_duration_hours"])

In [24]:
features = ["log_stay_duration", "hour_of_day"]

In [25]:
df["target"].mean()

np.float64(0.7422955593302596)

In [26]:
df["target_12h"] = (df["time_to_next"] <= 12).astype(int)
df["target_12h"].mean()

np.float64(0.5923319582625576)

In [39]:
df["target_6h"] = (df["time_to_next"] <= 6).astype(int)
df["target_6h"].mean()

np.float64(0.5633962264150943)

In [40]:
df["target_8h"] = (df["time_to_next"] <= 8).astype(int)
df["target_8h"].mean()

np.float64(0.5830188679245283)